## Get the German temperture data, using Berlin as a representative example

In [1]:
!pip install requests



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# Project paths
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from electricity_demand.data import load_processed_data

# Weekly load data
weekly = load_processed_data()            # index is weekly dates, column 'load_gw'
load = weekly["load_gw"]                  # just a Series if you need it

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Weekly head:")
print(weekly.head())
print("Date range:", load.index.min(), "to", load.index.max())

PROJECT_ROOT: /Users/indhureddy/Documents/electricity-demand-forecasting
Weekly head:
                             load_gw
date                                
2015-01-04 00:00:00+00:00  47.233740
2015-01-11 00:00:00+00:00  56.191101
2015-01-18 00:00:00+00:00  57.672679
2015-01-25 00:00:00+00:00  58.613304
2015-02-01 00:00:00+00:00  58.734030
Date range: 2015-01-04 00:00:00+00:00 to 2020-10-04 00:00:00+00:00


In [9]:
# Make weekly index timezone-naive to match temp_weekly_raw
weekly.index = weekly.index.tz_convert(None)

print("Weekly index sample after tz_convert(None):", weekly.index[:3])

Weekly index sample after tz_convert(None): DatetimeIndex(['2015-01-04', '2015-01-11', '2015-01-18'], dtype='datetime64[us]', name='date', freq=None)


In [4]:
def get_open_meteo_temperature(
    latitude=52.52,
    longitude=13.41,
    start_date="2015-01-01",
    end_date="2020-12-31",
):
    """
    Download daily mean temperature from Open-Meteo archive API.
    Berlin is used by default.
    """
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "temperature_2m_mean",
        "timezone": "Europe/Berlin",
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()["daily"]

    temp = pd.DataFrame({
        "date": pd.to_datetime(data["time"]),
        "temperature_2m_mean": data["temperature_2m_mean"],
    })

    temp = temp.set_index("date")
    return temp


# Fetch daily temperature over the load date range
temp_daily = get_open_meteo_temperature(
    start_date=str(load.index.min().date()),
    end_date=str(load.index.max().date()),
)

print("Temp_daily head:")
print(temp_daily.head())

Temp_daily head:
            temperature_2m_mean
date                           
2015-01-04                  3.0
2015-01-05                  2.7
2015-01-06                  1.2
2015-01-07                  2.1
2015-01-08                  3.1


## Convert to a weekly feature

In [10]:
# 1. Build weekly temp features from resample (using resample's own index)
temp_weekly_raw = pd.DataFrame({
    "temp_mean": temp_daily["temperature_2m_mean"].resample("W").mean(),
    "temp_min":  temp_daily["temperature_2m_mean"].resample("W").min(),
    "temp_max":  temp_daily["temperature_2m_mean"].resample("W").max(),
})

base_heat = 15.5
base_cool = 22.0

temp_weekly_raw["heating_degree"] = (
    np.maximum(base_heat - temp_daily["temperature_2m_mean"], 0)
    .resample("W")
    .sum()
)

temp_weekly_raw["cooling_degree"] = (
    np.maximum(temp_daily["temperature_2m_mean"] - base_cool, 0)
    .resample("W")
    .sum()
)

print("temp_weekly_raw index sample:", temp_weekly_raw.index[:5])
print("temp_weekly_raw head:")
print(temp_weekly_raw.head())

# 2. Align to weekly load index
temp_weekly = temp_weekly_raw.reindex(weekly.index)

print("Temp_weekly aligned head:")
print(temp_weekly.head())

temp_weekly_raw index sample: DatetimeIndex(['2015-01-04', '2015-01-11', '2015-01-18', '2015-01-25',
               '2015-02-01'],
              dtype='datetime64[us]', name='date', freq='W-SUN')
temp_weekly_raw head:
            temp_mean  temp_min  temp_max  heating_degree  cooling_degree
date                                                                     
2015-01-04   3.000000       3.0       3.0            12.5             0.0
2015-01-11   3.885714       1.2       8.5            81.3             0.0
2015-01-18   4.900000      -0.8       9.2            74.2             0.0
2015-01-25   0.028571      -0.7       0.9           108.3             0.0
2015-02-01   1.414286      -0.1       2.8            98.6             0.0
Temp_weekly aligned head:
            temp_mean  temp_min  temp_max  heating_degree  cooling_degree
date                                                                     
2015-01-04   3.000000       3.0       3.0            12.5             0.0
2015-01-11   3.8

## Merge with e.g. the electricity load data

In [11]:
# Merge with electricity load data

# Start from existing weekly load DataFrame
feature_df = weekly.copy()  # has 'load_gw' and weekly index

# Join temperature features
feature_df = feature_df.join(temp_weekly)

print("After join, head:")
print(feature_df.head())
print("Shape after join:", feature_df.shape)

# Interpolate and drop NaNs
feature_df = feature_df.interpolate("time")
print("NaN counts after interpolate:")
print(feature_df.isna().sum())

feature_df = feature_df.dropna()
print("Final head after dropna:")
print(feature_df.head())
print("Final shape:", feature_df.shape)

After join, head:
              load_gw  temp_mean  temp_min  temp_max  heating_degree  \
date                                                                   
2015-01-04  47.233740   3.000000       3.0       3.0            12.5   
2015-01-11  56.191101   3.885714       1.2       8.5            81.3   
2015-01-18  57.672679   4.900000      -0.8       9.2            74.2   
2015-01-25  58.613304   0.028571      -0.7       0.9           108.3   
2015-02-01  58.734030   1.414286      -0.1       2.8            98.6   

            cooling_degree  
date                        
2015-01-04             0.0  
2015-01-11             0.0  
2015-01-18             0.0  
2015-01-25             0.0  
2015-02-01             0.0  
Shape after join: (301, 6)
NaN counts after interpolate:
load_gw           0
temp_mean         0
temp_min          0
temp_max          0
heating_degree    0
cooling_degree    0
dtype: int64
Final head after dropna:
              load_gw  temp_mean  temp_min  temp_max  heati

In [12]:
feature_df.to_csv(
    PROJECT_ROOT / "data" / "processed" / "weekly_load_temperature_features.csv"
)

check_df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "weekly_load_temperature_features.csv"
)

print("Check_df head:")
print(check_df.head())
print(check_df.shape)

Check_df head:
         date    load_gw  temp_mean  temp_min  temp_max  heating_degree  \
0  2015-01-04  47.233740   3.000000       3.0       3.0            12.5   
1  2015-01-11  56.191101   3.885714       1.2       8.5            81.3   
2  2015-01-18  57.672679   4.900000      -0.8       9.2            74.2   
3  2015-01-25  58.613304   0.028571      -0.7       0.9           108.3   
4  2015-02-01  58.734030   1.414286      -0.1       2.8            98.6   

   cooling_degree  
0             0.0  
1             0.0  
2             0.0  
3             0.0  
4             0.0  
(301, 7)
